# 01 — Diagnosis Kualitas Data (M2)

Memeriksa **lima dimensi** kualitas data **sebelum** membersihkan apa pun. Data asli dibaca dari `../data/raw/` dan tidak diubah.

**Dataset:** laptopPrice.csv (harga dan spesifikasi laptop)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

# Path RELATIF terhadap folder notebooks/ (tidak ada path lokal seperti C:\Users)
RAW = Path('../data/raw/laptopPrice.csv')
PROCESSED = Path('../data/processed/laptop_bersih.csv')
FIG = Path('../reports/figures'); FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW)
print('Ukuran:', df.shape)
df.info()

In [ ]:
df.head()

## 1. Kelengkapan
Nilai kosong **dan** placeholder yang menyamar sebagai nilai (`Not Available`).

In [ ]:
kosong = df.isna().sum()
print('NaN per kolom (hanya yang > 0):'); print(kosong[kosong > 0] if kosong.any() else 'tidak ada NaN')

n_gen_na = (df['processor_gnrtn'] == 'Not Available').sum()
print(f"\nprocessor_gnrtn = 'Not Available': {n_gen_na} baris ({n_gen_na/len(df):.1%})")

In [ ]:
# Pola kekosongan: apakah 'Not Available' berkaitan dengan kolom lain?
pola = (df.assign(gen_na = df['processor_gnrtn'] == 'Not Available')
          .groupby('processor_name')['gen_na'].agg(['sum', 'count']))
pola['persen'] = (pola['sum'] / pola['count'] * 100).round(1)
pola.sort_values('persen', ascending=False)

**Tafsiran:** `Not Available` hampir seluruhnya muncul pada keluarga prosesor tertentu (Ryzen, Celeron, Pentium) — polanya **terstruktur** (penamaan generasi tidak berlaku untuk merek itu), bukan acak. Maka sebaiknya **ditandai**, bukan diisi/ditebak.

## 2. Keunikan

In [ ]:
n_dup = df.duplicated().sum()
print(f'Baris duplikat persis (salinan ke-2 dst): {n_dup} ({n_dup/len(df):.1%})')
print(f'Baris yang terlibat duplikasi: {df.duplicated(keep=False).sum()}')
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(8)

## 3. Konsistensi
Tipe data dan penulisan nilai.

In [ ]:
print(df.dtypes)
print('\nKolom bertipe teks tetapi isinya angka + satuan:')
for c in ['ram_gb','ssd','hdd','graphic_card_gb','warranty','rating']:
    print(f'  {c:16s} contoh: {df[c].unique()[:4].tolist()}')
print('\nPenulisan merek:', sorted(df['brand'].unique()))

**Tafsiran:** kolom spesifikasi berupa teks (`'8 GB'`, `'4 stars'`) sehingga tidak bisa dihitung; penulisan merek tidak seragam (`acer`, `DELL`, `APPLE` vs `Lenovo`, `Avita`).

## 4. Validitas
Nilai yang secara logika tidak masuk akal.

In [ ]:
nol = df['Number of Ratings'] == 0
print('Number of Ratings = 0              :', nol.sum(), 'baris')
print('...dan distribusi kolom rating-nya :'); print(df.loc[nol, 'rating'].value_counts())
print('\nUntuk perbandingan, rating pada baris yang punya penilai:'); print(df.loc[~nol, 'rating'].value_counts().sort_index())

**Tafsiran:** seluruh baris tanpa penilai berisi `3 stars` — itu **nilai bawaan (placeholder)**, bukan penilaian nyata. Jika dibiarkan, rata-rata rating terseret ke 3.

In [ ]:
print('Number of Reviews > Number of Ratings:', (df['Number of Reviews'] > df['Number of Ratings']).sum())
print('Price <= 0:', (df['Price'] <= 0).sum())
ram = df['ram_gb'].str.replace(' GB', '', regex=False).astype(int)
bentrok = (df['os_bit'] == '32-bit') & (ram >= 8)
print('OS 32-bit dengan RAM >= 8 GB (janggal):', bentrok.sum())

## 5. Akurasi / kewajaran
Outlier harga — dicek dahulu sebelum diputuskan.

In [ ]:
d = df.drop_duplicates()
q1, q3 = d['Price'].quantile([.25, .75]); batas = q3 + 1.5 * (q3 - q1)
out = d[d['Price'] > batas]
print(f'Batas atas IQR: {batas:,.0f} | outlier: {len(out)} baris ({len(out)/len(d):.1%})')
print('Skewness harga:', round(d['Price'].skew(), 2))
print('\nMerek pada outlier:'); print(out['brand'].value_counts())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(d['Price'], bins=40, ax=ax[0]); ax[0].set_title('Sebaran harga (menjulur ke kanan)')
sns.boxplot(x=d['Price'], ax=ax[1]); ax[1].set_title('Boxplot harga')
plt.tight_layout(); plt.savefig(FIG / 'quality_harga.png', dpi=120); plt.show()

**Tafsiran:** cek merek pada outlier di atas; periksa juga spesifikasinya (RAM, prosesor). Bila didominasi spesifikasi tinggi, outlier **wajar** — ditandai, tidak dibuang.

## Ringkasan temuan

In [ ]:
profil = pd.DataFrame([
 ['Keunikan',   'seluruh kolom',      'Baris duplikat persis',                       int(df.duplicated().sum()),                'Sedang', 'Hapus, simpan satu'],
 ['Kelengkapan','processor_gnrtn',    "Placeholder 'Not Available' (terstruktur)",   int((df['processor_gnrtn']=='Not Available').sum()), 'Sedang', 'Jadikan kosong + kolom penanda'],
 ['Validitas',  'rating',             "'3 stars' bawaan pada baris tanpa penilai",   int(nol.sum()),                            'Tinggi', 'Jadikan kosong'],
 ['Validitas',  'os_bit & ram_gb',    'OS 32-bit dengan RAM >= 8 GB',                int(bentrok.sum()),                        'Rendah', 'Tandai, jangan ditimpa'],
 ['Konsistensi','brand',              'Huruf besar/kecil tidak seragam',             int(df['brand'].isin(['acer','DELL','APPLE']).sum()), 'Rendah', 'Seragamkan'],
 ['Tipe Data',  'ram_gb, ssd, hdd, graphic_card_gb, warranty, rating', 'Angka tersimpan sebagai teks bersatuan', len(df), 'Tinggi', 'Konversi ke angka'],
 ['Akurasi',    'Price',              'Outlier harga (IQR)',                         len(out),                                  'Rendah', 'Tandai, jangan dibuang'],
], columns=['dimensi','kolom','masalah','jumlah','keparahan','rencana'])
profil['persen'] = (profil['jumlah'] / len(df) * 100).round(2)
profil.to_csv('../reports/profil_kualitas_laptop.csv', index=False)
profil